# 02 - Préparation et feature engineering

On adapte le kernel Kaggle [LightGBM with Simple Features](https://www.kaggle.com/code/jsaguiar/lightgbm-with-simple-features/script) de Aguiar pour produire un dataset enrichi prêt à la modélisation.

**Ce qu'on garde du kernel :** la structure modulaire (une fonction par table), les agrégations numériques (min/max/mean/sum/var), les splits Active/Closed pour le Bureau et Approved/Refused pour les précédentes demandes, les ratios métier (DAYS_EMPLOYED_PERC, PAYMENT_RATE, etc.).

**Ce qu'on change :**
- on ne charge pas `application_test.csv` (pas de soumission Kaggle, on fera notre split train/test au moment de la modélisation)
- on supprime tout le code lié à l'entraînement LightGBM (ce sera dans le notebook 03)
- on sauvegarde le dataset final en parquet (plus rapide et compact que CSV pour ce volume)

Au final on obtient un seul DataFrame avec ~307k lignes et ~800 colonnes, jointes via `SK_ID_CURR`.

## Choix méthodologiques

Quelques décisions importantes pour la suite :

- **Pas de suppression de colonnes** même avec beaucoup de NaN : certaines sont nos meilleures features (`EXT_SOURCE_1` a >50% de NaN mais c'est la 3e variable la plus corrélée à la cible).
- **Pas d'imputation ici** : elle se fera dans un `Pipeline` scikit-learn lors de l'entraînement, pour éviter le data leakage en cross-validation (la médiane d'imputation doit se calculer uniquement sur le fold de train).
- **Encodage :** `factorize` pour les variables binaires (`CODE_GENDER`, `FLAG_OWN_CAR`, `FLAG_OWN_REALTY`) et **one-hot encoding** pour les autres catégorielles.
- **Pas de chargement de `application_test.csv`** : c'est le test set Kaggle (sans `TARGET`), inutilisable pour évaluer notre modèle. On fera notre propre `train_test_split` au moment de la modélisation.
- **Format de sortie : parquet** : plus rapide à lire/écrire et plus compact que CSV pour ~800 colonnes.

In [1]:
import numpy as np
import pandas as pd
import gc
import time
from contextlib import contextmanager

import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/raw/'
OUTPUT_PATH = '../data/processed/dataset.parquet'

# Mode débug : si True, charge seulement 10 000 lignes par table pour tester rapidement
DEBUG = False

## Helpers

Deux utilitaires repris du kernel :
- `timer` : context manager pour chronométrer chaque étape
- `one_hot_encoder` : fait un one-hot encoding sur toutes les colonnes de type `object` ou `str`. Le paramètre `nan_as_category` permet de créer une colonne dédiée aux NaN (utile pour les tables historiques où le fait que la valeur soit manquante peut porter du signal après agrégation).

In [2]:
@contextmanager
def timer(title):
    t0 = time.time()
    yield
    print(f"{title} - done in {time.time() - t0:.0f}s")

def one_hot_encoder(df, nan_as_category=True):
    original_columns = list(df.columns)
    # Pandas 3 utilise le dtype 'str' au lieu de 'object' pour les chaînes,
    # on inclut les deux pour être compatible
    categorical_columns = list(df.select_dtypes(include=['object', 'str']).columns)
    df = pd.get_dummies(df, columns=categorical_columns, dummy_na=nan_as_category)
    new_columns = [c for c in df.columns if c not in original_columns]
    return df, new_columns

## 1. Table principale : `application_train`

On encode les variables catégorielles, on corrige l'anomalie `DAYS_EMPLOYED` repérée dans l'EDA, et on crée quelques ratios métier (taux d'endettement, ratio annuité/crédit, etc.) qui ont du sens pour prédire un risque de défaut.

In [3]:
def application(num_rows=None, nan_as_category=False):
    df = pd.read_csv(DATA_DIR + 'application_train.csv', nrows=num_rows)
    print(f"Samples : {len(df)}")
    
    # On retire les 4 lignes avec CODE_GENDER == 'XNA' (valeurs aberrantes)
    df = df[df['CODE_GENDER'] != 'XNA']
    
    # Encodage binaire (0/1) pour les variables à 2 modalités
    for bin_feature in ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']:
        df[bin_feature], _ = pd.factorize(df[bin_feature])
    
    # One-hot encoding pour les autres catégorielles
    df, cat_cols = one_hot_encoder(df, nan_as_category)
    
    # Anomalie DAYS_EMPLOYED = 365243 -> NaN
    df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)
    
    # Features ratios
    df['DAYS_EMPLOYED_PERC'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    df['INCOME_CREDIT_PERC'] = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
    df['ANNUITY_INCOME_PERC'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    
    return df

## 2. `bureau` et `bureau_balance`

Ces deux tables contiennent les crédits du client auprès d'autres institutions financières. On agrège d'abord `bureau_balance` (historique mensuel) au niveau de chaque crédit (`SK_ID_BUREAU`), puis on agrège le tout au niveau du client (`SK_ID_CURR`).

On crée en plus deux groupes de features spécifiques : un pour les crédits **actifs** et un pour les crédits **clos**, ce qui aide le modèle à distinguer la situation actuelle de l'historique.

In [4]:
def bureau_and_balance(num_rows=None, nan_as_category=True):
    bureau = pd.read_csv(DATA_DIR + 'bureau.csv', nrows=num_rows)
    bb = pd.read_csv(DATA_DIR + 'bureau_balance.csv', nrows=num_rows)
    bb, bb_cat = one_hot_encoder(bb, nan_as_category)
    bureau, bureau_cat = one_hot_encoder(bureau, nan_as_category)
    
    # Agrégation de bureau_balance au niveau de chaque crédit
    bb_aggregations = {'MONTHS_BALANCE': ['min', 'max', 'size']}
    for col in bb_cat:
        bb_aggregations[col] = ['mean']
    bb_agg = bb.groupby('SK_ID_BUREAU').agg(bb_aggregations)
    bb_agg.columns = pd.Index([e[0] + "_" + e[1].upper() for e in bb_agg.columns.tolist()])
    bureau = bureau.join(bb_agg, how='left', on='SK_ID_BUREAU')
    bureau.drop(['SK_ID_BUREAU'], axis=1, inplace=True)
    del bb, bb_agg
    gc.collect()
    
    # Agrégations numériques au niveau du client
    num_aggregations = {
        'DAYS_CREDIT': ['min', 'max', 'mean', 'var'],
        'DAYS_CREDIT_ENDDATE': ['min', 'max', 'mean'],
        'DAYS_CREDIT_UPDATE': ['mean'],
        'CREDIT_DAY_OVERDUE': ['max', 'mean'],
        'AMT_CREDIT_MAX_OVERDUE': ['mean'],
        'AMT_CREDIT_SUM': ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_DEBT': ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_OVERDUE': ['mean'],
        'AMT_CREDIT_SUM_LIMIT': ['mean', 'sum'],
        'AMT_ANNUITY': ['max', 'mean'],
        'CNT_CREDIT_PROLONG': ['sum'],
        'MONTHS_BALANCE_MIN': ['min'],
        'MONTHS_BALANCE_MAX': ['max'],
        'MONTHS_BALANCE_SIZE': ['mean', 'sum']
    }
    cat_aggregations = {}
    for cat in bureau_cat: cat_aggregations[cat] = ['mean']
    for cat in bb_cat: cat_aggregations[cat + "_MEAN"] = ['mean']
    
    bureau_agg = bureau.groupby('SK_ID_CURR').agg({**num_aggregations, **cat_aggregations})
    bureau_agg.columns = pd.Index(['BURO_' + e[0] + "_" + e[1].upper() for e in bureau_agg.columns.tolist()])
    
    # Crédits actifs : agrégations numériques uniquement
    active = bureau[bureau['CREDIT_ACTIVE_Active'] == 1]
    active_agg = active.groupby('SK_ID_CURR').agg(num_aggregations)
    active_agg.columns = pd.Index(['ACTIVE_' + e[0] + "_" + e[1].upper() for e in active_agg.columns.tolist()])
    bureau_agg = bureau_agg.join(active_agg, how='left', on='SK_ID_CURR')
    del active, active_agg
    gc.collect()
    
    # Crédits clos : agrégations numériques uniquement
    closed = bureau[bureau['CREDIT_ACTIVE_Closed'] == 1]
    closed_agg = closed.groupby('SK_ID_CURR').agg(num_aggregations)
    closed_agg.columns = pd.Index(['CLOSED_' + e[0] + "_" + e[1].upper() for e in closed_agg.columns.tolist()])
    bureau_agg = bureau_agg.join(closed_agg, how='left', on='SK_ID_CURR')
    del closed, closed_agg, bureau
    gc.collect()
    
    return bureau_agg

## 3. `previous_application`

Demandes de crédit précédentes du client chez Home Credit. Même logique que pour le Bureau : on crée un ratio (`APP_CREDIT_PERC` = montant demandé / montant accordé), puis on agrège globalement et on ajoute deux groupes de features spécifiques : demandes **approuvées** vs **refusées**.

In [5]:
def previous_applications(num_rows=None, nan_as_category=True):
    prev = pd.read_csv(DATA_DIR + 'previous_application.csv', nrows=num_rows)
    prev, cat_cols = one_hot_encoder(prev, nan_as_category=True)
    
    # Code 365243 = NaN (même anomalie que dans application)
    prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan, inplace=True)
    prev['DAYS_FIRST_DUE'].replace(365243, np.nan, inplace=True)
    prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan, inplace=True)
    prev['DAYS_LAST_DUE'].replace(365243, np.nan, inplace=True)
    prev['DAYS_TERMINATION'].replace(365243, np.nan, inplace=True)
    
    # Ratio montant demandé / montant reçu
    prev['APP_CREDIT_PERC'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT']
    
    # Agrégations numériques
    num_aggregations = {
        'AMT_ANNUITY': ['min', 'max', 'mean'],
        'AMT_APPLICATION': ['min', 'max', 'mean'],
        'AMT_CREDIT': ['min', 'max', 'mean'],
        'APP_CREDIT_PERC': ['min', 'max', 'mean', 'var'],
        'AMT_DOWN_PAYMENT': ['min', 'max', 'mean'],
        'AMT_GOODS_PRICE': ['min', 'max', 'mean'],
        'HOUR_APPR_PROCESS_START': ['min', 'max', 'mean'],
        'RATE_DOWN_PAYMENT': ['min', 'max', 'mean'],
        'DAYS_DECISION': ['min', 'max', 'mean'],
        'CNT_PAYMENT': ['mean', 'sum'],
    }
    cat_aggregations = {}
    for cat in cat_cols:
        cat_aggregations[cat] = ['mean']
    
    prev_agg = prev.groupby('SK_ID_CURR').agg({**num_aggregations, **cat_aggregations})
    prev_agg.columns = pd.Index(['PREV_' + e[0] + "_" + e[1].upper() for e in prev_agg.columns.tolist()])
    
    # Demandes approuvées
    approved = prev[prev['NAME_CONTRACT_STATUS_Approved'] == 1]
    approved_agg = approved.groupby('SK_ID_CURR').agg(num_aggregations)
    approved_agg.columns = pd.Index(['APPROVED_' + e[0] + "_" + e[1].upper() for e in approved_agg.columns.tolist()])
    prev_agg = prev_agg.join(approved_agg, how='left', on='SK_ID_CURR')
    
    # Demandes refusées
    refused = prev[prev['NAME_CONTRACT_STATUS_Refused'] == 1]
    refused_agg = refused.groupby('SK_ID_CURR').agg(num_aggregations)
    refused_agg.columns = pd.Index(['REFUSED_' + e[0] + "_" + e[1].upper() for e in refused_agg.columns.tolist()])
    prev_agg = prev_agg.join(refused_agg, how='left', on='SK_ID_CURR')
    
    del refused, refused_agg, approved, approved_agg, prev
    gc.collect()
    return prev_agg

## 4. `POS_CASH_balance`

Historique mensuel des crédits POS et cash chez Home Credit. On agrège les indicateurs de retard (`SK_DPD`, `SK_DPD_DEF`) et la durée d'historique (`MONTHS_BALANCE`).

In [6]:
def pos_cash(num_rows=None, nan_as_category=True):
    pos = pd.read_csv(DATA_DIR + 'POS_CASH_balance.csv', nrows=num_rows)
    pos, cat_cols = one_hot_encoder(pos, nan_as_category=True)
    
    aggregations = {
        'MONTHS_BALANCE': ['max', 'mean', 'size'],
        'SK_DPD': ['max', 'mean'],
        'SK_DPD_DEF': ['max', 'mean']
    }
    for cat in cat_cols:
        aggregations[cat] = ['mean']
    
    pos_agg = pos.groupby('SK_ID_CURR').agg(aggregations)
    pos_agg.columns = pd.Index(['POS_' + e[0] + "_" + e[1].upper() for e in pos_agg.columns.tolist()])
    pos_agg['POS_COUNT'] = pos.groupby('SK_ID_CURR').size()
    
    del pos
    gc.collect()
    return pos_agg

## 5. `installments_payments`

Historique de remboursement (un paiement = une ligne). On dérive deux features importantes :
- `PAYMENT_PERC` / `PAYMENT_DIFF` : est-ce que le client a payé le montant attendu ?
- `DPD` / `DBD` : nombre de jours de retard ou d'avance sur chéance

Ce sont des indicateurs directs de la fiabilité de remboursement.

In [7]:
def installments_payments(num_rows=None, nan_as_category=True):
    ins = pd.read_csv(DATA_DIR + 'installments_payments.csv', nrows=num_rows)
    ins, cat_cols = one_hot_encoder(ins, nan_as_category=True)
    
    # Pourcentage et différence payé vs attendu
    ins['PAYMENT_PERC'] = ins['AMT_PAYMENT'] / ins['AMT_INSTALMENT']
    ins['PAYMENT_DIFF'] = ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT']
    
    # Jours de retard (DPD) et jours d'avance (DBD)
    ins['DPD'] = ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']
    ins['DBD'] = ins['DAYS_INSTALMENT'] - ins['DAYS_ENTRY_PAYMENT']
    ins['DPD'] = ins['DPD'].apply(lambda x: x if x > 0 else 0)
    ins['DBD'] = ins['DBD'].apply(lambda x: x if x > 0 else 0)
    
    aggregations = {
        'NUM_INSTALMENT_VERSION': ['nunique'],
        'DPD': ['max', 'mean', 'sum'],
        'DBD': ['max', 'mean', 'sum'],
        'PAYMENT_PERC': ['max', 'mean', 'sum', 'var'],
        'PAYMENT_DIFF': ['max', 'mean', 'sum', 'var'],
        'AMT_INSTALMENT': ['max', 'mean', 'sum'],
        'AMT_PAYMENT': ['min', 'max', 'mean', 'sum'],
        'DAYS_ENTRY_PAYMENT': ['max', 'mean', 'sum']
    }
    for cat in cat_cols:
        aggregations[cat] = ['mean']
    
    ins_agg = ins.groupby('SK_ID_CURR').agg(aggregations)
    ins_agg.columns = pd.Index(['INSTAL_' + e[0] + "_" + e[1].upper() for e in ins_agg.columns.tolist()])
    ins_agg['INSTAL_COUNT'] = ins.groupby('SK_ID_CURR').size()
    
    del ins
    gc.collect()
    return ins_agg

## 6. `credit_card_balance`

Historique mensuel des cartes de crédit chez Home Credit. Agrégation simple : on calcule min/max/mean/sum/var sur toutes les colonnes numériques.

In [8]:
def credit_card_balance(num_rows=None, nan_as_category=True):
    cc = pd.read_csv(DATA_DIR + 'credit_card_balance.csv', nrows=num_rows)
    cc, cat_cols = one_hot_encoder(cc, nan_as_category=True)
    
    cc.drop(['SK_ID_PREV'], axis=1, inplace=True)
    cc_agg = cc.groupby('SK_ID_CURR').agg(['min', 'max', 'mean', 'sum', 'var'])
    cc_agg.columns = pd.Index(['CC_' + e[0] + "_" + e[1].upper() for e in cc_agg.columns.tolist()])
    cc_agg['CC_COUNT'] = cc.groupby('SK_ID_CURR').size()
    
    del cc
    gc.collect()
    return cc_agg

## 7. Orchestration et sauvegarde

On enchaîne toutes les fonctions, on joint les agrégations à la table principale via `SK_ID_CURR`, et on sauvegarde le résultat en parquet.

In [9]:
num_rows = 10000 if DEBUG else None

with timer("Application"):
    df = application(num_rows)
    print("Shape :", df.shape)

with timer("Bureau et bureau_balance"):
    bureau = bureau_and_balance(num_rows)
    print("Bureau shape :", bureau.shape)
    df = df.join(bureau, how='left', on='SK_ID_CURR')
    del bureau
    gc.collect()

with timer("Previous applications"):
    prev = previous_applications(num_rows)
    print("Previous shape :", prev.shape)
    df = df.join(prev, how='left', on='SK_ID_CURR')
    del prev
    gc.collect()

with timer("POS-CASH balance"):
    pos = pos_cash(num_rows)
    print("POS shape :", pos.shape)
    df = df.join(pos, how='left', on='SK_ID_CURR')
    del pos
    gc.collect()

with timer("Installments payments"):
    ins = installments_payments(num_rows)
    print("Installments shape :", ins.shape)
    df = df.join(ins, how='left', on='SK_ID_CURR')
    del ins
    gc.collect()

with timer("Credit card balance"):
    cc = credit_card_balance(num_rows)
    print("Credit card shape :", cc.shape)
    df = df.join(cc, how='left', on='SK_ID_CURR')
    del cc
    gc.collect()

print("\nDataset final :", df.shape)

Samples : 307511
Shape : (307507, 247)
Application - done in 2s
Bureau shape : (305811, 116)
Bureau et bureau_balance - done in 10s
Previous shape : (338857, 249)
Previous applications - done in 10s
POS shape : (337252, 18)
POS-CASH balance - done in 6s
Installments shape : (339587, 26)
Installments payments - done in 12s
Credit card shape : (103558, 141)
Credit card balance - done in 7s

Dataset final : (307507, 797)


In [10]:
# Sauvegarde en parquet
with timer("Sauvegarde parquet"):
    df.to_parquet(OUTPUT_PATH, index=False)
    print(f"Sauvegardé dans : {OUTPUT_PATH}")

Sauvegardé dans : ../data/processed/dataset.parquet
Sauvegarde parquet - done in 6s


## 8. Validation de la sortie

Quelques vérifications rapides pour s'assurer que le dataset est bien formé.

In [11]:
print(f"Shape : {df.shape}")
print(f"Mémoire : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nDistribution de TARGET :")
print(df['TARGET'].value_counts(dropna=False))
print(f"\nNombre de colonnes avec au moins un NaN : {(df.isnull().sum() > 0).sum()}")
print(f"Pourcentage moyen de NaN : {df.isnull().mean().mean() * 100:.1f}%")
print(f"\nAperçu :")
df.head()

Shape : (307507, 797)
Mémoire : 1717.1 MB

Distribution de TARGET :
TARGET
0    282682
1     24825
Name: count, dtype: int64

Nombre de colonnes avec au moins un NaN : 614
Pourcentage moyen de NaN : 25.9%

Aperçu :


,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,CC_NAME_CONTRACT_STATUS_Signed_MAX,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_Signed_SUM,CC_NAME_CONTRACT_STATUS_Signed_VAR,CC_NAME_CONTRACT_STATUS_nan_MIN,CC_NAME_CONTRACT_STATUS_nan_MAX,CC_NAME_CONTRACT_STATUS_nan_MEAN,CC_NAME_CONTRACT_STATUS_nan_SUM,CC_NAME_CONTRACT_STATUS_nan_VAR,CC_COUNT
0,100002,1,0,0,0,0,202500.0,406597.5,24700.5,351000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0,1,0,1,0,270000.0,1293502.5,35698.5,1129500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0,0,1,0,0,67500.0,135000.0,6750.0,135000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0,1,0,0,0,135000.0,312682.5,29686.5,297000.0,...,False,0.0,0.0,0.0,False,False,0.0,0.0,0.0,6.0
4,100007,0,0,0,0,0,121500.0,513000.0,21865.5,513000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Récapitulatif

Le dataset final contient :
- une ligne par client (`SK_ID_CURR`)
- la cible `TARGET` (uniquement pour les clients de `application_train`)
- les features de la table principale + les agrégations issues des 6 tables annexes

Les valeurs manquantes sont conservées telles quelles : elles seront imputées au moment de l'entraînement via un `Pipeline` scikit-learn pour éviter le data leakage en cross-validation.

On passe au notebook 03 pour la modélisation.